# Text to speech
Text to speech systems (TTS) transforms written text into a waveform with the corresponding spoken text.

There exist a set of commercial or free products for this task, e.g.
eleven labs https://beta.elevenlabs.io/ (last visited on June 2023) or
Thorsten TTS https://www.thorsten-voice.de/ (last visited on November 2024).

One of the simplest ways to use text to speech in python is the module pyttsx3:

https://pypi.org/project/pyttsx3/

This module is a wrapper for the os-internal text to speech system. At least for the operating system Windows, the results are satisfying.

In [1]:
import pyttsx3
import time
import whisper
import torch
import numpy as np
import sounddevice as sd
import soundfile as sf
import matplotlib.pyplot as plt
import librosa
import sys
sys.path.insert(1, '../Basics')
import WaveFileHandling

In [2]:
Filename = 'output.wav'

# initialize the speech synthesizer
engine = pyttsx3.init()

rate = engine.getProperty('rate')   # getting details of current speaking rate
engine.setProperty('rate', 125)     # setting up new voice rate

# https://ichi.pro/de/eine-einfuhrung-in-pyttsx3-ein-text-zu-sprache-konverter-fur-python-81905511310787
voices = engine.getProperty('voices')       #getting details of current voice
for voice in voices:
    print("Voice: %s" % voice.name)
    print(" - ID: %s" % voice.id)
    print(" - Languages: %s" % voice.languages)
    print(" - Gender: %s" % voice.gender)
    print(" - Age: %s" % voice.age)
    print("\n")

engine.setProperty('voice', voices[0].id)  #changing index, changes voices. 0 for german female
#engine.setProperty('voice', voices[1].id)   #changing index, changes voices. 1 for english female

# select the language
engine.setProperty('voice', 'german')

# select the text
sentence = "Lassen Sie die Ente zu Wasser."

StartTime = time.time()
# use the synthesizer to say the sentence
#engine.say(sentence)
# use the synthesizer to save the wave form to a file
engine.save_to_file(sentence, Filename)

# wait until everything is finished
engine.runAndWait()

StopTime = time.time()

Voice: Microsoft Hedda Desktop - German
 - ID: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_DE-DE_HEDDA_11.0
 - Languages: []
 - Gender: None
 - Age: None


Voice: Microsoft Zira Desktop - English (United States)
 - ID: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_EN-US_ZIRA_11.0
 - Languages: []
 - Gender: None
 - Age: None




## Latency and real-time capability
In order to measure the real-time capability of the algorithm on the used hardware, the length of the output waveform and the duration of evaluation is measured.

If the length of the output waveform is longer than the duraction of evaluation, the algorithm is real-time capable.

In [3]:
x, Fs = WaveFileHandling.ReadWaveAsNumpyArray(Filename)

print(str(x.shape[0] / Fs), ' seconds of audio generated in ', str(StopTime - StartTime), ' seconds')

3.075691609977324  seconds of audio generated in  0.16317510604858398  seconds


# Speech to text
Modern speech to text systems are based on transformer networks.

The model whisper is very simple to install and to use. It can work without any web connectivity.

The only disadvantage of the whisper is the relatively high computational complexity.

## Scientific publication
Publication of the whisper: Robust Speech Recognition via Large-ScaleWeak Supervision

## Installation
The installation can be done in the following three steps:

1) install ffmpeg

2) install git: https://learn.microsoft.com/en-us/devops/develop/git/install-and-set-up-git

3) install whisper: pip install git+https://github.com/openai/whisper.git

or simply by

pip install -U openai-whisper

## Configuration
The whisper can use different languages and model sizes:

<table>
<tr><td>Size</td><td>Parameters</td><td>English-only model</td><td>Multilingual model</td><td>Required VRAM</td><td>Relative speed</td></tr>
<tr><td>tiny</td><td>39 M</td><td>tiny.en</td><td>tiny</td><td>~1 GB</td><td>~32x</td></tr>
<tr><td>base</td><td>74 M</td><td>base.en</td><td>base</td><td>~1 GB</td><td>~16x</td></tr>
<tr><td>small</td><td>244 M</td><td>small.en</td><td>small</td><td>~2 GB</td><td>~6x</td></tr>
<tr><td>medium</td><td>769 M</td><td>medium.en</td><td>medium</td><td>~5 GB</td><td>~2x</td></tr>
<tr><td>large</td><td>1550 M</td><td>N/A</td><td>large</td><td>~10 GB</td><td>1x</td></tr>
</table>
Source: https://github.com/openai/whisper


In [4]:
ModelSize = "small" # ["tiny", "base", "small", "medium", "large"]
Language = 'German' # ["German", "English"]

## Calling the whisper
The whisper can be started by the following code:

In [5]:
SamplingRateWhisper = 16000 # the sampling rate should always be 16 kHz
devices = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # source: https://github.com/openai/whisper/discussions/373
model = whisper.load_model(ModelSize, device = devices)  

def DoResampling(x, OriginalSamplingRate):
    if np.abs(SamplingRateWhisper - OriginalSamplingRate) > 1:
        y = librosa.resample(x, orig_sr = OriginalSamplingRate, target_sr = SamplingRateWhisper)
        y *= (np.max(np.abs(x)) / np.max(np.abs(y)))
        return y
    else:
        return x

def CallWhisper(x, OriginalSamplingRate):
    y = DoResampling(x, OriginalSamplingRate)
    result = model.transcribe(y.astype(np.float32), language=Language, verbose=0)
    return result["text"]

c:\Users\spiertz\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\__init__.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp,

## Demo
In the following, a small example for the usage of the whisper is shown.

The so called real time factor is computational time or running time divided by the audio length. A real time factor below 1 means, that the algorithm is capable of working in real time.

The latency is the length of the audio recording plus the running time. The latency measurement starts with the beginning of the audio recording and ends with printing out the result.

In [6]:
x, Fs = WaveFileHandling.ReadWaveAsNumpyArray(Filename)

start = time.time()
result = CallWhisper(x, Fs)
stop = time.time()
print(result)
RunTime = stop-start
AudioLength = x.shape[0] / Fs
RealTimeFactor = RunTime / AudioLength
print("runtime: " + str(RunTime) + " s")
print("audio length:" + str(AudioLength) + " s")
print("real time factor: " + str(RealTimeFactor))
print("latency: " + str(RunTime + AudioLength) + " s")

c:\Users\spiertz\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Lassen sie die Ente zu Wasser.
runtime: 9.02674412727356 s
audio length:3.075691609977324 s
real time factor: 2.934866453447883
latency: 12.102435737250884 s


# Levenshtein Distance
In a speech to text scenario (STT) three different kind of errors may occur:

1) Insertion of a letter: kitten -> kiatten

2) Deletion of a letter: kitten -> kiten

3) Substitution of a letter: kitten -> kisten

The Levenshtein distance is a simple algorithm used for evaluating the distance between two different strings by adding up the insertions, deletions and substitutions.

A detailled description of the Levenshtein distance can be found under:

https://en.wikipedia.org/wiki/Levenshtein_distance

In the following, two implementations of the Levenshtein distance are introduced:

For better understanding, the first implementation is described in detail:

The Levenshtein distance is implemented by a distance matrix, through which the shortest path of errors should be found.
The three error types can be described by movements in the distance matrix:
- A movement to the right direction corresponds to an additional deletion from string 1 to string 2 and increases the distance by one.
- A movement to the down direction corresponds to an additional insertion from string 1 to string 2 and increases the distance by one.
- A diagonal movement is either an substitution, if both positions in both strings are not identical or a correct letter. In the case of an substitution, the distance is increased by one.

The algorithm can be described by the following:
- Aim: The Levenshtein distance between string $s_1$ of length $R-1$ and string $s_2$ of length $C-1$ should be evaluated.
- Step 0: For simpler implementation, extend both strings with an identical symbol, here '#' is used.
- Step 1: Initialize a distance matrix $D$ with $R$ rows and $C$ columns. All elements are initialized by zeros.
- Step 2: Initialize the first row by increasing numbers: $D(0, c) = D(0, c - 1) + 1$ for $0<c<C$. All movements to the right direction corresponds to deletions from string 1 to string 2.
- Step 3: Initialize the first column by increasing numbers: $D(r, 0) = D(r - 1, 0) + 1$ for $0<r<R$. All movements to the down direction corresponds to insertions from string 1 to string 2.
- Step 4: For all other positions in the distance matrix $D(r,c)$ with $0<r<R$ and $0<c<C$ evaluate the distances $d_i$ from the three possible predecessors:
    - $d_0 = D(r-1, c) + 1$ assumes an insertion from string 1 to string 2, because it is a movement downwards.
    - $d_1 = D(r, c-1) + 1$ assumes a deletion from string 1 to string 2, because it is a movement to the right.
    - $d_2 = D(r-1,c-1) + d_x$ with $d_x=1$ assuming a substition for the case $s_1(r)!=s_2(c)$ and $d_x=0$ assumes identical symbols at the current position $s_1(r)=s_2(c)$.
    - Evaluate the local distance $D(r, c)$ by taking the minimum of $d_0$, $d_1$ and $d_2$.
- Step 5: The final Levenshtein distance is the element in the lower right corner of the distance matrix: $D(R-1, C-1)$

This implementation of the Levenshtein distance is compared to the reference implementation in the python package Levenshtein.

In [7]:
from Levenshtein import distance as lev

# implementation by the lab team
def LevenshteinDistance(s1, s2):
    # definition of an identical starting position, in order to find correct
    # insertions and deletions at the starting positions of the strings
    s1 = '#' + str(s1)
    s2 = '#' + str(s2)
    DistanceMatrix = np.zeros((len(s1), len(s2)))
    DistanceMatrix[0, :] = np.arange(DistanceMatrix.shape[1])
    DistanceMatrix[:, 0] = np.arange(DistanceMatrix.shape[0])
    Predecessor = np.zeros((3))
    for row in range(1, DistanceMatrix.shape[0]):        
        for col in range(1, DistanceMatrix.shape[1]):
            Predecessor[0] = DistanceMatrix[row-1, col] + 1 # insertion
            Predecessor[1] = DistanceMatrix[row, col-1] + 1 # deletion
            Predecessor[2] = DistanceMatrix[row-1, col-1] 
            if not (s1[row] == s2[col]):
                Predecessor[2] += 1 # substitution
            DistanceMatrix[row, col] = np.amin(Predecessor) # evaluate the updated cost for this position
    return DistanceMatrix[-1, -1]

# example
s1 = "kitten"
s2 = "skitting"
d1 = LevenshteinDistance(s1, s2)
print(f"The Levenshtein-Distance between '{s1}' and '{s2}' is {d1}.")
# test
d0 = lev(s1, s2) # reference implementation from the python-Levenshtein package
assert np.abs(d1-d0)<1e-1, 'error in evaluation of levenshtein distance'

The Levenshtein-Distance between 'kitten' and 'skitting' is 3.0.


## Programming Exercise
The Levenshtein distance so far is evaluated letter by letter.

It is possible to evaluate a Word Error Rate with insertions, deletions and replacements with the Levenshtein distance.

For this, the input strings must be splitted into single words.

After that, the Levenshtein distance can be applied to the single words using the above given algorithms.

Treat sentence signs as single words.

In [8]:
### solution begins
import re

def custom_split(text):
    text = '#' + text
    # Verwende re.split mit einem regulären Ausdruck, das Whitespaces und Satzzeichen einfängt
    return re.findall(r'\w+|[^\w\s]', text)
### solution ends

def CountWordErrors(s1, s2):
    ### solution begins
    s1 = custom_split(s1)
    s2 = custom_split(s2)
    DistanceMatrix = np.zeros((len(s1), len(s2)))
    DistanceMatrix[0, :] = np.arange(DistanceMatrix.shape[1])
    DistanceMatrix[:, 0] = np.arange(DistanceMatrix.shape[0])
    Predecessor = np.zeros((3))
    for row in range(1, DistanceMatrix.shape[0]):        
        for col in range(1, DistanceMatrix.shape[1]):
            Predecessor[0] = DistanceMatrix[row-1, col] + 1 # insertion
            Predecessor[1] = DistanceMatrix[row, col-1] + 1 # deletion
            Predecessor[2] = DistanceMatrix[row-1, col-1] 
            if not (s1[row] == s2[col]):
                Predecessor[2] += 1 # substitution
            DistanceMatrix[row, col] = np.amin(Predecessor) # evaluate the updated cost for this position
    WER = DistanceMatrix[-1, -1]
    ### solution ends
    return WER

import unittest

class TestProgrammingExercise(unittest.TestCase):

    def test_WER1(self):
        s1 = 'Pferde schnauben nicht die Nase'
        s2 = 'Nashörner schnauben selten nicht die'
        self.assertEqual(CountWordErrors(s1, s2), 3)
    
    def test_DealingWithTailingSigns(self):
        s1 = 'Pferde schnauben nicht die Nase.'
        s2 = 'Nashörner schnauben selten nicht die.'
        self.assertEqual(CountWordErrors(s1, s2), 3)

    def test_DealingWithIntermediateSigns(self):
        s1 = 'Ich denke, also bin ich.'
        s2 = 'Ich fahre also bin ich.'
        self.assertEqual(CountWordErrors(s1, s2), 2)
    
unittest.main(argv=[''], verbosity=2, exit=False)

test_DealingWithIntermediateSigns (__main__.TestProgrammingExercise.test_DealingWithIntermediateSigns) ... ok
test_DealingWithTailingSigns (__main__.TestProgrammingExercise.test_DealingWithTailingSigns) ... ok
test_WER1 (__main__.TestProgrammingExercise.test_WER1) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


## Exam Preparation
1) Determine the Levenshtein distance (letter based) between the two strings s1='I want you for president.' and s2='I wanna u 4 present.'. How many deletions, insertions and substitions occur.

2) Is the levenshtein distance ignoring additional blanks in the case of letter error rate or in the case of word error rate?

3) If the reference word is analysed over the different columns of the matrix for evaluating the Levenshtein distance and the recorded word is analysed over the different rows, what direction in the distance matrix corresponds to Insertions?

4) A transcription of $30$ seconds of audio takes $28$ seconds to transcribe to a text. Evaluate the latency and the real time factor. Is the algorithm capable of working in real time?

5) For real time appclications in speech to text, a latency of $100$ ms up to $200$ ms is reasonable. According to your experiments in this jupyter notebook: is the whisper usable in such a real time application?

6) Which sampling rate is necessary for the whisper?

7) Explain the meaning of the python code 'y *= (np.max(np.abs(x)) / np.max(np.abs(y)))' in the procedure DoResampling.

8) A simple text to speech (TTS) system can be implemented by recording all words spoken by a single person. With this set of recordings a TTS can be defined by concatenating the recordings of the target words. What problems arise with this implementation?

9) The probability of words in a language can be estimated by the Zipf theorem: If you consider the $N$ words with highest probability, the probability of the word with position $n$ can be evaluated by $p(n)\approx\frac{1}{n\cdot\ln\left(1.78\cdot N\right)}$. Evaluate the probabilities of the five words with highest probability 'der', 'die', 'und', 'in' and 'den' according to 'Häufigkeitswörterbuch von F. W. Kaeding, 1897' assuming a set of words of $N=10^5$. Evaluate in python: How many words do you need, in order to represent $50$ % of the words used in a language according to Zipf theorem.

10) Evaluate the Levenshtein distance between the two strings 'Kisten' and 'Karst'. Evaluation means: evaluate the whole distance matrix by the above given algorithm.

In [9]:
### solution task 9
TargetProbability = 0.5
NumberOfWordsInALanguage = 1e5
sum = 0.0
n = 1
while sum < (np.log(1.78*NumberOfWordsInALanguage) * TargetProbability):
    sum += 1/n
    n += 1
print(n, ' words have a probability of ', TargetProbability)


238  words have a probability of  0.5


## Summary
After working with this Jupyter Notebook you should be able to explain the following topics:

- Name a sample tool for text to speech applications.
- Name a sample tool for speech to text applications.
- Explain the Levenshtein distance by your own words.
- Evaluate the Levenshtein distance for two given strings.